# Homework <a class="anchor" id="section_homework"></a>

**HW 2/1**: <br>
Smart up our plotter to be able to customize the hard-coded layout setup inside the function. Add more custom feature that you would like to use.


**HW 2/2**: <br>
Upgrade the BSM call option pricer to be able to return the greeks as well, not just the price.


In [2]:
from typing import Union, List
import numpy as np
from scipy.stats import norm


def black_scholes_eur_call(
    r: float,
    T: float,
    S0: float,
    sigma: float,
    K: Union[float, List[float], np.ndarray],
):
    """
    Black-Scholes pricer of European call option on non-dividend-paying stock

    param r: risk-free interest rate (which is constant)
    param T: time to maturity (in years)
    param S0: initial spot price of the underlying stock
    param sigma: volatility of the underlying stock
    param 
    """
    #check conditions
    assert sigma > 0

    K_vec = np.array([K]) if isinstance(K, float) else np.array(K)

    d1_vec = (np.log(S0 / K_vec) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2_vec = d1_vec - sigma * np.sqrt(T)

    
    N_d1_vec = norm.cdf(d1_vec)
    N_d2_vec = norm.cdf(d2_vec)
    n_d1_vec = norm.pdf(d1_vec)  #sűrűségfüggvény ()

    #PRICE
    price = N_d1_vec*S0 - K_vec * np.exp((-1.0)*r*T) * N_d2_vec

    #GREEKS
    delta = N_d1_vec

    gamma = n_d1_vec / (S0 * sigma * np.sqrt(T))

    vega = S0 * n_d1_vec * np.sqrt(T)

    theta = (
        -S0 * n_d1_vec * sigma / (2 * np.sqrt(T))
        - r * K_vec * np.exp(-r * T) * N_d2_vec
    )

    rho = K_vec * T * np.exp(-r * T) * N_d2_vec

    # dictionary a return
    return {
        "price": price,
        "delta": delta,
        "gamma": gamma,
        "vega": vega,
        "theta": theta,
        "rho": rho,
    }

In [5]:
call=black_scholes_eur_call(r=0.05, T=1, S0=20.0, sigma=0.3, K=10)
print("price: ", call["price"])
print("delta: ", call["delta"])

price:  10.49652216075794
delta:  0.9956949235179967


**HW 2/3**: <br>
Implement the BSM pricer for Eluropean put option.


In [6]:
def black_scholes_eur_put(
    r: float,
    T: float,
    S0: float,
    sigma: float,
    K: Union[float, List[float], np.ndarray],
):
    """
    Black-Scholes pricer of European call option on non-dividend-paying stock

    param r: risk-free interest rate (which is constant)
    param T: time to maturity (in years)
    param S0: initial spot price of the underlying stock
    param sigma: volatility of the underlying stock
    param 
    """
    #check conditions
    assert sigma > 0

    K_vec = np.array([K]) if isinstance(K, float) else np.array(K)

    d1_vec = (np.log(S0 / K_vec) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2_vec = d1_vec - sigma * np.sqrt(T)

    
    N_d1_vec = norm.cdf(d1_vec)
    N_d2_vec = norm.cdf(d2_vec)
    n_d1_vec = norm.pdf(d1_vec)  #sűrűségfüggvény ()

    #PRICE
    price = K_vec * np.exp(-r*T) * norm.cdf(-d2_vec) - S0 * norm.cdf(-d1_vec)

    #GREEKS
    delta = norm.cdf(d1_vec) - 1

    gamma = n_d1_vec / (S0 * sigma * np.sqrt(T))

    vega = S0 * n_d1_vec * np.sqrt(T)

    theta = (
        -S0 * n_d1_vec * sigma / (2 * np.sqrt(T))
        + r * K_vec * np.exp(-r*T) * norm.cdf(-d2_vec)
    )

    rho = -K_vec * T * np.exp(-r*T) * norm.cdf(-d2_vec)

    # dictionary a return
    return {
        "price": price,
        "delta": delta,
        "gamma": gamma,
        "vega": vega,
        "theta": theta,
        "rho": rho,
    }

In [7]:
put=black_scholes_eur_put(r=0.05, T=1, S0=20.0, sigma=0.3, K=10)
put["price"]

0.008816405765081262

**HW 2/4**: <br>
With the call and put pricer, check if Put-Call parity holds in practice.

\begin{equation}
c_t - p_t = S_t - K\cdot e^{-r\cdot(T-t)}
\end{equation}

In [8]:
r=0.05
T=1
S0=100
sigma=0.3
K=105

call=black_scholes_eur_call(r,T,S0,sigma,K)
put=black_scholes_eur_put(r,T,S0,sigma,K)
call["price"]-put["price"]-(S0-K*np.exp(-r*T))  #csak kerekítési hiba miatt nem nulla

-1.4210854715202004e-14